# RAG en pratique — Chatbot qui connaît des destinations voyage

**Objectif :** Construire un chatbot qui répond aux questions sur des destinations en se basant sur des documents réels (pas son entraînement)  
**Stack :** LangChain + Chroma + Embeddings **local** (FastEmbed) + Google Gemini  
**Durée :** 45 min

---

### Ce qu'on va construire

```
Fichiers texte (Maroc, Japon, Islande)
        │
        ▼
   1. Charger (Loader)
        │
        ▼
   2. Découper (Splitter)
        │
        ▼
   3. Vectoriser (Embeddings LOCAL — pas de clé !)
        │
        ▼
   4. Stocker (Chroma)
        │
        ▼
   5. Chercher + Répondre (RAG Chain + Gemini)
```

### Prérequis
- Clé API Google Gemini (gratuite → [aistudio.google.com](https://aistudio.google.com/)) — **uniquement pour la réponse finale**
- Python 3.10+

> 💡 **Bonne nouvelle :** les étapes 1 à 5 (le cœur du RAG) fonctionnent **100% en local sans clé API**. Seule la génération de la réponse (étape 6) utilise Gemini.

---

## Étape 0 — Installation

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters chromadb fastembed python-dotenv
print("Installation terminée ✓")

---

## Étape 1 — Charger la clé API

Créez un fichier `.env` à la racine du projet avec :
```
GOOGLE_API_KEY=AIzaXXXXXXXXXXXXXXXXXXXXXXXXX
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print("Clé API :", "✅ chargée" if GOOGLE_API_KEY else "❌ manquante — vérifiez .env")

---

## Étape 2 — Charger les documents (Document Loaders)

On charge les 3 fichiers texte du dossier `data/`. LangChain les transforme en objets `Document`.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data/",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"📄 {len(documents)} documents chargés :")
for doc in documents:
    print(f"   - {doc.metadata['source']} ({len(doc.page_content)} caractères)")

In [ ]:
# Voir à quoi ressemble un Document
print("=== Premier document ===")
print(f"Source : {documents[0].metadata['source']}")
print(f"Contenu (100 premiers caractères) : {documents[0].page_content[:100]}...")

---

## Étape 3 — Découper en chunks (Text Splitter)

Les textes sont trop longs pour être envoyés tels quels. On les découpe en morceaux avec un **chevauchement** pour ne pas perdre de contexte.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(documents)

print(f"✂️  {len(documents)} documents → {len(chunks)} chunks")
print()
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} (source: {chunk.metadata['source']}) ---")
    print(chunk.page_content[:150], "...")
    print()

---

## Étape 4 — Créer les embeddings + stocker dans Chroma

On transforme chaque chunk en **vecteur numérique** (embedding) et on stocke le tout dans **ChromaDB** (base vectorielle locale).

> 🔑 **Pas de clé API ici !** On utilise **FastEmbed**, un modèle d'embeddings qui tourne **en local** sur votre machine. Le premier lancement télécharge le modèle (~130 Mo), ensuite tout est instantané et hors-ligne. On choisit un modèle **multilingue** car nos textes sont en français.

In [ ]:
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

# Modèle d'embeddings LOCAL et MULTILINGUE (pas de clé API)
embeddings = FastEmbedEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"✅ {len(chunks)} chunks vectorisés et stockés dans ChromaDB")

---

## Étape 5 — Tester la recherche vectorielle

Avant de construire le RAG complet, testons que la recherche fonctionne. On pose une question et Chroma retourne les chunks les plus pertinents.

In [ ]:
question = "Quel budget prévoir pour voyager au Japon ?"

resultats = vectorstore.similarity_search(question, k=3)

print(f"🔍 Question : {question}\n")
for i, doc in enumerate(resultats):
    print(f"--- Résultat {i+1} (source: {doc.metadata['source']}) ---")
    print(doc.page_content[:200])
    print()

In [ ]:
# Test avec score de similarité
resultats_avec_score = vectorstore.similarity_search_with_score(question, k=3)

print("📊 Scores de similarité (plus bas = plus pertinent) :\n")
for doc, score in resultats_avec_score:
    source = doc.metadata['source'].split('/')[-1]
    print(f"   {source:15} → score: {score:.4f}")
    print(f"   {doc.page_content[:80]}...")
    print()

---

## Étape 6 — Construire la chaîne RAG complète

On assemble le pipeline : **question → recherche → prompt enrichi → LLM → réponse**.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Le LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3
)

# 2. Le retriever (chercheur de documents pertinents)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. Le prompt template RAG
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Tu es un agent de voyage expert. "
     "Réponds UNIQUEMENT à partir du contexte fourni ci-dessous. "
     "Si l'information n'est pas dans le contexte, dis-le honnêtement. "
     "Réponds en français, sois précis et donne les chiffres quand disponibles."),
    ("human", 
     "Contexte :\n{context}\n\n"
     "Question : {question}")
])

# 4. Fonction pour formater les documents en texte
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 5. Assembler la chaîne RAG avec LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Chaîne RAG assemblée et prête !")

---

## Étape 7 — Tester le RAG !

C'est le moment de vérité. On pose des questions et le chatbot répond **à partir des documents**, pas de sa mémoire.

In [ ]:
# Test 1 — Question sur le Japon
question = "Quel budget prévoir pour voyager au Japon ?"
reponse = rag_chain.invoke(question)

print(f"❓ {question}\n")
print(f"🤖 {reponse}")

In [ ]:
# Test 2 — Question sur le Maroc
question = "Quels sont les incontournables de Marrakech ?"
reponse = rag_chain.invoke(question)

print(f"❓ {question}\n")
print(f"🤖 {reponse}")

In [ ]:
# Test 3 — Question sur l'Islande
question = "Quand peut-on voir les aurores boréales en Islande ?"
reponse = rag_chain.invoke(question)

print(f"❓ {question}\n")
print(f"🤖 {reponse}")

In [ ]:
# Test 4 — Question HORS contexte (le RAG doit dire qu'il ne sait pas)
question = "Quelle est la capitale de l'Australie ?"
reponse = rag_chain.invoke(question)

print(f"❓ {question}\n")
print(f"🤖 {reponse}")
print()
print("👆 Le RAG devrait avouer qu'il n'a pas l'info — c'est le comportement attendu !")

---

## Étape 8 — Comparer : avec RAG vs sans RAG

Pour prouver que le RAG utilise bien les documents et pas ses connaissances générales.

In [ ]:
question = "Combien coûte l'entrée au Blue Lagoon en Islande ?"

# AVEC RAG (utilise nos documents)
reponse_rag = rag_chain.invoke(question)

# SANS RAG (appel direct au LLM, sans contexte)
reponse_directe = llm.invoke(question)

print("=" * 60)
print(f"❓ Question : {question}")
print("=" * 60)
print(f"\n🟢 AVEC RAG (basé sur nos docs) :\n{reponse_rag}")
print(f"\n🔵 SANS RAG (connaissances générales du LLM) :\n{reponse_directe.content}")
print("\n💡 La réponse RAG cite le prix exact de nos docs (75€) !")

---

## Étape 9 — Mini chatbot RAG interactif

Un chatbot en boucle pour poser plusieurs questions d'affilée.

In [ ]:
print("🌍 Chatbot Voyage RAG")
print("Posez vos questions sur le Maroc, le Japon ou l'Islande.")
print("Tapez 'quit' pour quitter.\n")

while True:
    question = input("Vous : ")
    if question.lower() in ["quit", "exit", "q"]:
        print("Au revoir ! 👋")
        break

    reponse = rag_chain.invoke(question)
    print(f"\n🤖 Bot : {reponse}\n")

---

## Étape 10 — Exercice : ajoutez votre propre destination !

**À vous de jouer :**

1. Créez un fichier `data/votre_pays.txt` avec des infos sur une destination de votre choix
2. Relancez les cellules depuis l'étape 2
3. Posez des questions sur cette nouvelle destination

**Questions à tester :**
- Le chatbot répond-il correctement sur votre nouvelle destination ?
- Que se passe-t-il si vous posez une question dont la réponse n'est dans aucun fichier ?

---

## Récapitulatif — Ce qu'on a construit

```
data/*.txt                          ← Nos données (3 fichiers voyage)
    │
    ▼
DirectoryLoader + TextLoader        ← Étape 2 : charger
    │
    ▼
RecursiveCharacterTextSplitter      ← Étape 3 : découper (chunks de 500 chars)
    │
    ▼
FastEmbedEmbeddings (LOCAL)         ← Étape 4 : vectoriser (sans clé API)
    │
    ▼
Chroma (./chroma_db)                ← Étape 4 : stocker
    │
    ▼
retriever | format_docs             ← Étape 6 : chercher les chunks pertinents
    │
    ▼
ChatPromptTemplate                  ← Étape 6 : injecter le contexte dans le prompt
    │
    ▼
ChatGoogleGenerativeAI              ← Étape 6 : générer la réponse (Gemini)
    │
    ▼
StrOutputParser                     ← Étape 6 : extraire le texte
```

### Les imports utilisés

```python
# Loaders
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings LOCAL + Vector Store
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

# LLM + Chain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
```

### À retenir
- Le **RAG** permet à un LLM de répondre à partir de **vos données**, pas de son entraînement
- Les **embeddings peuvent tourner en local** (FastEmbed) — pas besoin de clé pour vectoriser
- **Chroma** stocke les vecteurs localement (pas besoin de cloud)
- La **chaîne LCEL** `retriever | prompt | llm | parser` est le pattern standard
- Le **prompt système** est crucial : "réponds UNIQUEMENT à partir du contexte"
- Tester avec une question **hors contexte** vérifie que le RAG ne hallucine pas